## Listes de 9 questions

Quand et comment les clients commandent-ils généralement ?  

Quels produits faut-il mettre en avant le samedi ?  

Quels autres produits les clients commandent-ils fréquemment avec du chocolat ?  

Quels sont les principaux profils de clients : acheteurs nocturnes, acheteurs occasionnels ou accros au  panier ?

Quel article les clients ajoutent-ils en premier à leur panier ?

Quels produits ont le plus de chances d’être commandés à nouveau ?

Les clients commandent-ils en moyenne plus de nourriture pour chats le matin que de nourriture pour chiens le soir ?

Existe-t-il une corrélation entre l’heure de la dernière commande et la probabilité de recommander un produit ?

Peut-on prédire si un produit sera commandé à nouveau par un client lors de sa prochaine visite ?

Importation des modules

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Ouverture des fichiers 

In [4]:
df_aisles = pd.read_csv("./datasets/aisles.csv")
df_departments = pd.read_csv("./datasets/departments.csv")
df_order_products = pd.read_csv("./datasets/order_products.csv")
df_orders = pd.read_csv("./datasets/orders.csv")
df_products = pd.read_csv("./datasets/products.csv")

Display que les 10 premières lignes

In [8]:
display(df_aisles.head(10))
display(df_departments.head(10))
display(df_order_products.head(10))
display(df_orders.head(10))
display(df_products.head(10))

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation
5,6,other
6,7,packaged meat
7,8,bakery desserts
8,9,pasta sauce
9,10,kitchen supplies


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol
5,6,international
6,7,beverages
7,8,pets
8,9,dry goods pasta
9,10,bulk


,order_id,product_id,add_to_cart_order,reordered
0,2,33120.0,1.0,1.0
1,2,28985.0,2.0,1.0
2,2,9327.0,3.0,0.0
3,2,45918.0,4.0,1.0
4,2,30035.0,5.0,0.0
5,2,17794.0,6.0,1.0
6,2,40141.0,7.0,1.0
7,2,1819.0,8.0,1.0
8,2,43668.0,9.0,0.0
9,4,46842.0,1.0,0.0


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1.0,1.0,2.0,8.0,NaN
1,2398795,1.0,2.0,3.0,7.0,15.0
2,473747,1.0,3.0,3.0,12.0,21.0
3,2254736,1.0,4.0,4.0,7.0,29.0
4,431534,1.0,5.0,4.0,15.0,28.0
5,2295261,1.0,9.0,1.0,16.0,0.0
6,2550362,1.0,10.0,4.0,8.0,30.0
7,2168274,2.0,1.0,2.0,11.0,NaN
8,839880,2.0,14.0,3.0,10.0,13.0
9,444309,3.0,2.0,3.0,19.0,9.0


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13
5,6,Dry Nose Oil,11,11
6,7,Pure Coconut Water With Orange,98,7
7,8,Cut Russet Potatoes Steam N' Mash,116,1
8,9,Light Strawberry Blueberry Yogurt,120,16
9,10,Sparkling Orange Juice & Prickly Pear Beverage,115,7


Vérifications qu'il n'y a pas de valeurs nul.

In [ ]:
display(df_aisles.isnull().sum())
display(df_departments.isnull().sum())
display(df_order_products.isnull().sum())
display(df_orders.isnull().sum())
display(df_products.isnull().sum())

Affichage des dimension de chaque csv

In [10]:
display(df_aisles.shape)
display(df_departments.shape)
display(df_order_products.shape)
display(df_orders.shape)
display(df_products.shape)

(134, 2)

(21, 2)

(13692886, 4)

(1444444, 6)

(49688, 4)

## Quand et comment les clients commandent-ils généralement ?  

In [ ]:
df_orders_hour = df_orders["order_hour_of_day"].value_counts().sort_index()
df_orders_day = df_orders["days_since_prior_order"].value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.plot(df_orders_hour.index, df_orders_hour.values, marker="o", color="blue")
plt.title("Pic de commandes dans la journée")
plt.xlabel("Heures de la journée")
plt.ylabel("Commandes")
plt.show()

>Le pic matinal : L'affluence maximale se situe autour de 10h du matin. Les clients ont donc tendance à organiser leurs courses en ligne en milieu de matinée plutôt que le soir.

In [ ]:
df_orders_hour = df_orders["order_hour_of_day"].value_counts().sort_index()
df_orders_day = df_orders["days_since_prior_order"].value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.plot(df_orders_day.index, df_orders_day.values, marker="o", color="blue")
plt.title("Pic de commandes dans le mois")
plt.xlabel("Jour dans le mois")
plt.ylabel("Commandes")
plt.show()

>La routine hebdomadaire : On observe des rebonds de commandes très nets tous les 7, 14 et 21 jours. Cela met en lumière une routine bien ancrée chez les familles, qui effectuent leurs achats de manière très régulière, semaine après semaine.

>Le grand ravitaillement mensuel : Le pic le plus important se trouve à 30 jours. Il correspond aux clients qui font un gros "plein" une fois par mois (souvent après la paie), mais il représente aussi un plafond technique de la base de données qui regroupe toutes les commandes espacées de 30 jours et plus.

## Quels produits faut-il mettre en avant le samedi ?  

In [ ]:
df_orders_saturday = df_orders[df_orders["order_dow"] == 6]
df_sat_prod = pd.merge(df_orders_saturday, df_order_products, on="order_id")
df_final = pd.merge(df_sat_prod, df_products, on="product_id")
top_10_sat = df_final["product_name"].value_counts().head(10)
df_orders_sat = df_sat_prod["product_id"].value_counts().sort_index()


couleurs = plt.cm.inferno(np.linspace(0.3, 0.9, 10))


plt.figure(figsize=(10, 5))
top_10_sat.plot(kind="bar", color=couleurs)
plt.xticks(rotation=45, ha="right")
plt.title("Produits les plus achetés le samedi")
plt.xlabel("Nom des produits")
plt.ylabel("Commandes")
plt.show()

>Au vu des données, nous recommandons de mettre fortement en avant les bananes le samedi, car il y a plus de 25 000 commandes pour ce produit ce jour-là.

## Quels autres produits les clients commandent-ils fréquemment avec du chocolat ?  


In [ ]:
df_chocolat = df_products[df_products["product_name"].str.contains("Chocolate", na=False)]
df_chocolat_panier = pd.merge(df_chocolat, df_order_products, on="product_id")
df_chocolat_final = df_chocolat_panier["order_id"]
df_product_article = df_order_products[df_order_products["order_id"].isin(df_chocolat_final)]
df_nom_finaux = pd.merge(df_product_article, df_products, on="product_id")
df_final_all = pd.merge(df_nom_finaux, df_orders, on="order_id")

produit_avec_chocolat = df_nom_finaux["product_name"].value_counts().head(10)

couleurs = plt.cm.YlOrRd(np.linspace(0.9, 0.3, 10))

plt.figure(figsize=(10, 5))
produit_avec_chocolat.plot(kind="barh", color=couleurs)
plt.title("Top 10 des Produits les plus achetés avec du chocolat")
plt.xlabel("Nombre de commandes")
plt.ylabel("Produits")
plt.gca().invert_yaxis()
plt.show()

>On en conclut donc que, même en achetant du chocolat, les bananes dominent largement les paniers des clients.

>En conclusion, même lors d'un achat plaisir comme le chocolat, les bananes et les produits frais continuent de dominer largement le panier des clients.

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [ ]:
print(df_final_all.isna().sum())

In [ ]:
encoders = {}

le_product_name = LabelEncoder()
df_final_all["days_since_prior_order"] = df_final_all["days_since_prior_order"].fillna(0)
df_final_all = df_final_all.dropna(subset=["reordered"])

df_final_all["product_name"] = le_product_name.fit_transform(df_final_all["product_name"].astype(str))
X = df_final_all[["order_hour_of_day", "order_dow", "days_since_prior_order", "product_name", "add_to_cart_order"]]
y = df_final_all["reordered"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
modele = RandomForestClassifier(n_estimators=1, max_depth=1, n_jobs=1).fit(X=X_train, y=y_train)
y_pred = modele.predict(X_test)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
classification_report = classification_report(y_test, y_pred)
print(accuracy)
print(classification_report)

In [ ]:
joblib.dump(modele, "model.joblib")

joblib.dump(X_train.columns.tolist(), "model_columns.joblib")

joblib.dump(encoders, "encoders.joblib")